# Working with the GSDB

### Packages

In [1]:
import pandas as pd
import numpy as np
import json

### Loading Data and Basic Filtering

In [22]:
gsdb = pd.read_csv('GSDB_V4.csv')
gsdb = gsdb[(gsdb['trade'] == 1)]
gsdb = gsdb[( gsdb['begin'].isin(list(range(1991, 2017))) ) | ( gsdb['end'].isin(list(range(1991, 2017))) )]
# gsdb = gsdb[( gsdb['descr_trade'].str.contains('imp_compl') ) | ( gsdb['descr_trade'].str.contains('imp_part') )]
gsdb = gsdb.reset_index()
# gsdb['sanctioned_state'].unique()
# gsdb.groupby('descr_trade').size()
gsdb

,index,case_id,sanctioned_state,sanctioning_state,begin,end,trade,descr_trade,arms,military,financial,travel,other,target_mult,sender_mult,objective,success
0,11,12,Comecon,"Austria, Finland, Sweden, Switzerland",1950,1994,1,exp_part,0,0,0,0,0,1,1,"prevent_war,policy_change","success_total,failed"
1,12,13,Comecon,CoCom,1950,1994,1,exp_part,0,0,0,0,0,1,1,"prevent_war,policy_change","success_total,failed"
2,17,18,Palestine,League of Arab States,1950,1994,1,"exp_compl,imp_compl",0,0,0,0,0,0,1,territorial_conflict,failed
3,40,41,"Korea, North",United States,1955,2008,1,"exp_compl,imp_compl",0,0,1,0,0,0,0,prevent_war,failed
4,92,93,South Africa,Switzerland,1963,1994,1,exp_part,1,0,0,0,0,0,0,policy_change,success_total
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,1050,1051,"Egypt, Arab Rep.",Saudi Arabia,2016,2017,1,exp_part,0,0,0,0,0,0,0,democracy,failed
276,1052,1053,Iran,Australia,2016,2023,1,exp_part,1,1,1,1,0,0,0,prevent_war,ongoing
277,1053,1054,Iran,Canada,2016,2023,1,exp_part,0,0,1,0,0,0,0,prevent_war,ongoing
278,1055,1056,Iran,Switzerland,2016,2016,1,exp_part,0,0,0,1,0,0,0,prevent_war,success_total


### Fixing Names

In [23]:
with open('country_correction.json') as json_file:
    country_corrections = json.load(json_file)['countries']

def fill_empty_correction_schemes():
    for cnt in country_corrections:
        if cnt['alternatives'] == [] and cnt['scheme'] == '':
            cnt['alternatives'] = [cnt['standard_name']]
            cnt['scheme'] = 'or_in'

fill_empty_correction_schemes()

def satisfies_scheme(country, scheme, alternatives):
    if scheme == 'exact':
        return country.lower() in alternatives
    elif scheme == 'all_in':
        return [alt in country.lower() for alt in alternatives] == [True] * len(alternatives)
    else:
        return [alt in country.lower() for alt in alternatives] != [False] * len(alternatives)

def correct_country(country):
    standard_names = [d['standard_name'] for d in country_corrections]
    if country.lower() in standard_names:
        return country.title()
    else:
        country_object = next((country_obj for country_obj in country_corrections if satisfies_scheme(country.lower(), country_obj['scheme'], country_obj['alternatives'])), '')
        return country_object['standard_name'].title() if country_object else ''
    
def get_accompanying_code(country):
    country_object = next((country_obj for country_obj in country_corrections if country.lower() == country_obj['standard_name']), '')
    return country_object['code'] if country_object else ''

with open('exclusion_list.txt', 'r') as file:
    exclusion_lines = file.readlines()
    exclusion_lines = [line.rstrip('\n') for line in exclusion_lines]
exclusion_lines = exclusion_lines[1:]

In [24]:
def country_correct_dataset(df, column):
    df = df.rename(columns = {column: 'country'})

    df = df.query('country not in @exclusion_lines')
    df['country'] = df['country'].map(correct_country)
    df['code'] = df['country'].map(get_accompanying_code)
    df = df.sort_values('country')
    df = df[df['country'] != '']
    return df

### Exploding the Rows

In [3]:
gsdb['years'] = gsdb.apply(lambda x: list(np.arange(x['begin'], x['end'] + 1)), axis = 1)
gsdb_sploded_years = gsdb.explode('years').reset_index()
gsdb_sploded_years = gsdb_sploded_years[['case_id', 'sanctioned_state', 'sanctioning_state', 'years', 'descr_trade', 'objective', 'success', 'begin', 'end']]
gsdb_sploded_years = gsdb_sploded_years.rename(columns = {'years': 'year'})
gsdb_sploded_years = gsdb_sploded_years[gsdb_sploded_years['year'] > 1990]
gsdb_sploded_years
# gsdb_sploded['sanctioning_state'] = gsdb.apply(lambda x: gsdb_sploded['sanctioning_state'].split(', '))

,case_id,sanctioned_state,sanctioning_state,year,descr_trade,objective,success,begin,end
41,12,Comecon,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994
42,12,Comecon,"Austria, Finland, Sweden, Switzerland",1992,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994
43,12,Comecon,"Austria, Finland, Sweden, Switzerland",1993,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994
44,12,Comecon,"Austria, Finland, Sweden, Switzerland",1994,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994
86,13,Comecon,CoCom,1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994
...,...,...,...,...,...,...,...,...,...
2585,1058,"Korea, North",United States,2019,exp_compl,prevent_war,ongoing,2016,2023
2586,1058,"Korea, North",United States,2020,exp_compl,prevent_war,ongoing,2016,2023
2587,1058,"Korea, North",United States,2021,exp_compl,prevent_war,ongoing,2016,2023
2588,1058,"Korea, North",United States,2022,exp_compl,prevent_war,ongoing,2016,2023


### Correcting Countries

In [25]:
replacing_dict = {
    'South Vietnam': pd.NA, 'Transjordan': 'Jordan',
    'Yugoslavia': (', ').join(['Bosnia and Herzegovina', 'Croatia', 'Macedonia', 'Montenegro', 'Serbia', 'Slovenia', 'Kosovo']),
    'Congo, Democratic Republic of the': 'Democratic Republic of Congo', 'Egypt, Arab Rep.': 'Egypt', 'Ceylon': 'Sri Lanka', 'Korea, North': 'North Korea',
    'Comecon': (', ').join(['Bulgaria', 'Cuba', 'Czechia', 'Slovakia', 'Hungary', 'Mongolia', 'Poland', 'Romania', 'Vietnam']),
}

In [26]:
gsdb_sploded_years_2 = gsdb_sploded_years.copy()
gsdb_sploded_years_2['sanctioned_state'] = gsdb_sploded_years['sanctioned_state'].apply(lambda x: replacing_dict[x] if x in replacing_dict.keys() else x)
gsdb_sploded_years_2['sanctioned_state'] = gsdb_sploded_years_2['sanctioned_state'].apply(lambda x: str(x).split(', '))
gsdb_sploded = gsdb_sploded_years_2.explode('sanctioned_state')
gsdb_sploded['exp'] = gsdb_sploded['descr_trade'].str.contains('exp').astype(int)
gsdb_sploded['imp'] = gsdb_sploded['descr_trade'].str.contains('imp').astype(int)
gsdb_sploded

,case_id,sanctioned_state,sanctioning_state,year,descr_trade,objective,success,begin,end,exp,imp
41,12,Bulgaria,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994,1,0
41,12,Cuba,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994,1,0
41,12,Czechia,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994,1,0
41,12,Slovakia,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994,1,0
41,12,Hungary,"Austria, Finland, Sweden, Switzerland",1991,exp_part,"prevent_war,policy_change","success_total,failed",1950,1994,1,0
...,...,...,...,...,...,...,...,...,...,...,...
2585,1058,North Korea,United States,2019,exp_compl,prevent_war,ongoing,2016,2023,1,0
2586,1058,North Korea,United States,2020,exp_compl,prevent_war,ongoing,2016,2023,1,0
2587,1058,North Korea,United States,2021,exp_compl,prevent_war,ongoing,2016,2023,1,0
2588,1058,North Korea,United States,2022,exp_compl,prevent_war,ongoing,2016,2023,1,0


In [27]:
gsdb_sploded_named = country_correct_dataset(gsdb_sploded, 'sanctioned_state')
gsdb_sploded_named.to_csv('GSDB_Sploded.csv', index = False)

In [35]:
gsdb_sploded_corrected = gsdb_sploded.drop(['begin', 'end', 'objective'], axis = 1)
custom_aggergation_dict = {
    'sanctioning_state': (",").join,
    'descr_trade': (",").join,
    'success': (",").join,
    'exp': 'max', 'imp': 'max'
}

gsdb_sploded_corrected = gsdb_sploded.reset_index().groupby(['sanctioned_state', 'year']).agg(custom_aggergation_dict).reset_index()
gsdb_sploded_corrected
#gsdb_sploded_test.reset_index().groupby(['sanctioned_state', 'year']).max()

,sanctioned_state,year,sanctioning_state,descr_trade,success,exp,imp
0,Afghanistan,1999,United States,"exp_part,imp_part",success_part,1,1
1,Afghanistan,2000,"United States,UN","exp_part,imp_part,exp_part","success_part,failed",1,1
2,Afghanistan,2001,"United States,UN,EU, Cyprus, Malta, Turkey, Cr...","exp_part,imp_part,exp_part,exp_part","success_part,failed,failed,nego_settlement,failed",1,1
3,Afghanistan,2002,"United States,UN,EU, Cyprus, Malta, Turkey, Cr...","exp_part,imp_part,exp_part,exp_part","success_part,failed,failed,nego_settlement,failed",1,1
4,Albania,2015,Russia,imp_part,ongoing,0,1
...,...,...,...,...,...,...,...
1046,Zimbabwe,2023,"EU, Turkey, Croatia, Macedonia, Montenegro, Ic...","exp_part,exp_part,exp_part","ongoing,ongoing,ongoing,ongoing,ongoing,ongoing",1,0
1047,nan,1991,United States,"exp_compl,imp_compl",failed,1,1
1048,nan,1992,United States,"exp_compl,imp_compl",failed,1,1
1049,nan,1993,United States,"exp_compl,imp_compl",failed,1,1


In [36]:
gsdb_sploded_named_corrected = country_correct_dataset(gsdb_sploded_corrected, 'sanctioned_state')
gsdb_sploded_named_corrected.to_csv('GSDB_Sploded_Corrected.csv', index = False)